In [ ]:
import mujoco
import mujoco.viewer
import mediapy as media
import matplotlib.pyplot as plt
import time
import itertools
import numpy as np
import serial

In [ ]:
def map_val(x, in_min, in_max, out_min, out_max):
    return (x - in_min) * (out_max - out_min) / (in_max - in_min) + out_min

In [ ]:
#leg parameters
parameters = {
    "plane_len": 0.8,
    "thigh_len": 0.3, "thigh_size": 0.04, "thigh_mass": 2.0,
    "shank_len": 0.25, "shank_size": 0.03, "shank_mass": 1.5,
    "metatarsus_len": 0.15, "metatarsus_size": 0.025, "metatarsus_mass": 0.5
}

path = "xml/rr_bonilla.xml"

#mujoco model load
with open(path, "r") as file:
    xml_temp = file.read()

xml = xml_temp.format(**parameters)
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

In [ ]:
# Serial connection
#This port must be the same as the serial port used in Arduino code
#in this case is ACM0
try:
    #Baudrate must be the same as in the arduino code
    arduino = serial.Serial(port='/dev/ttyACM0', baudrate=115200, timeout=0.01)
    print("Conectado al Arduino correctamente.")
    time.sleep(2) # Wait for arduino reset
except Exception as e:
    print(f"Error conectando Arduino: {e}")
    print("La simulación correrá sin control externo.")
    arduino = None

In [ ]:
#control loop
print("Iniciando visor interactivo...")

with mujoco.viewer.launch_passive(model, data) as viewer:
    while viewer.is_running():
        step_start = time.time()

        # Reading arduino
        if arduino and arduino.in_waiting:
            try:
                linea = arduino.readline().decode().strip()
                
                # Verificar que la línea no esté vacía
                if linea:
                    valores = linea.split(',')
                    
                    # Asegurarnos de que recibimos exactamente 3 valores
                    if len(valores) == 3:
                        #This part of the code takes the serial values and breaks it into the info needed for each joint
                        pot1 = int(valores[0]) # Hip
                        pot2 = int(valores[1]) # Knee
                        pot3 = int(valores[2]) # Ankle
                        
                        # 2. MAP AND CONTROLL
                        # Adjust lectures to angles between -1 and 1 radians
                        # MuJoCo body has define angle ranges, they must be the same here
                        
                        # (Hip)
                        ctrl_hip = map_val(pot1, 0, 1023, -1.0, 1.0)
                        data.ctrl[0] = ctrl_hip
                        
                        # (Knee)
                        ctrl_knee = map_val(pot2, 0, 1023, -1.0, 1.0) 
                        data.ctrl[1] = ctrl_knee
                        
                        # (Ankle)
                        ctrl_ankle = map_val(pot3, 0, 1023, -0.5, 0.5)
                        data.ctrl[2] = ctrl_ankle
                        

            except ValueError:
                pass # some lines are corrupt, they must be ignored
            except Exception as e:
                print(f"Error de lectura: {e}")

        # Render MuJoCo simulation
        mujoco.mj_step(model, data)
        viewer.sync()

        # Time magement in simulation
        time_until_next_step = model.opt.timestep - (time.time() - step_start)
        if time_until_next_step > 0:
            time.sleep(time_until_next_step)

if arduino:
    arduino.close()